
When the DataLoader gathers samples from the dataset, they often come out inconsistent, and a batch must be a rectangular tensor. Common problems:

- **Different shapes:** variable sequence lengths (`[5, 8, 2]` vs `[7, 1, 9, 4, 6]`)
- **Different data types:** ints, floats, strings mixed across fields
- **Different number of items:** e.g. images with 2 vs 7 boxes
- **Missing or `None` fields:** samples without a label
- **Corrupted samples:** an image that fails to open, an empty text


So to solve that, we can make an function. The `collate_fn` receives the list of B samples and turns it into a uniform batch.


# 1. Collate

A function to modify and engineer the data present in the batch, for example:

- **Adjust:** fix types, filter invalid samples, fill missing fields
- **Resize:** pad or truncate sequences to a common length
- **Mask:** build the `attention_mask` and set `-100` on labels to be ignored by the loss
- **And more:** add extra fields (`position_ids`, `token_type_ids`), tokenize raw text, convert everything to tensors

After the batch gathers all the samples, we can use this to engineering the data and pass as a argument into the loader

```python

def collate_fn(batch):
    '''
    
    Logic here

    '''


loader = DataLoader(
    dataset = dataset,
    batch_size = X,
    collate_fn = collate_fn # HERE
)




```
---


## Input of collate

The collate_fn receives a **list of B samples**, one tuple per sample:

```python
batch = [
    ([5, 8, 2],       1),     # sample 1: (input_ids, label)
    ([7, 1, 9, 4, 6], 0),     # sample 2: (input_ids, label)
    ([3, 3],          1),     # sample 3: (input_ids, label)
]
```

It is called **once per batch**. Now we can apply all the logic.

---

## Receive the data

Split the samples by field: one list for x and one for`y.


**1**
```python
xs, ys = zip(*batch)
# xs = ([5, 8, 2], [7, 1, 9, 4, 6], [3, 3])
# ys = (1, 0, 1)
```

**2**
```python
xs = []
ys = []
for sample in batch:
    xs.append(sample[0])
    ys.append(sample[1])
# xs = ([5, 8, 2], [7, 1, 9, 4, 6], [3, 3])
# ys = (1, 0, 1)
```

**3**
```python
xs = [item[0] for item in batch]
ys = [item[1] for item in batch]
# xs = ([5, 8, 2], [7, 1, 9, 4, 6], [3, 3])
# ys = (1, 0, 1)
```

---

## Logic

Examples of what we can do with the data in the batch:

### 1. Padding

Complete the short sequences up to the longest one, so the batch becomes a rectangular tensor. The mask marks what is real (1) and what is padding (0).

```python
L = max(len(x) for x in xs)                       # longest in the batch (5)

x_pad = np.zeros((len(xs), L), dtype=np.int64)    # pad_id = 0
mask  = np.zeros((len(xs), L), dtype=np.int64)

for i, x in enumerate(xs):
    x_pad[i, :len(x)] = x                         # real tokens at the start of the row
    mask[i, :len(x)] = 1
```

```
x_pad:                 mask:
[5, 8, 2, 0, 0]        [1, 1, 1, 0, 0]
[7, 1, 9, 4, 6]        [1, 1, 1, 1, 1]
[3, 3, 0, 0, 0]        [1, 1, 0, 0, 0]
```

### 2. Truncation

Cut the sequences that exceed a `max_len`. It only removes tokens, so it needs no mask, but sequences still have different lengths afterwards, so **truncate first, then pad**.

```python
max_len = 4
xs = [x[:max_len] for x in xs]                    # shorter sequences stay unchanged
L = max(len(x) for x in xs)                       # now at most max_len
```

```
[5, 8, 2]        → [5, 8, 2]
[7, 1, 9, 4, 6]  → [7, 1, 9, 4]     (cut)
[3, 3]           → [3, 3]
```

Trade-off: truncation loses the end of long texts.

---

## Output of collate

A dict of **tensors**, all with `B` as the first dimension:

```python
return {
    'x': x_pad,      # (B, L)  int64
    'mask': mask,    # (B, L)  int64
    'y': y,          # (B,)    int64
}
```

The DataLoader returns exactly this, one batch per iteration:

```python
loader = DataLoader(ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

for batch in loader:
    x    = batch['x']
    mask = batch['mask']
    y    = batch['y']
```


## 1. Padding (NLP)

In [ ]:
import numpy as np

def collate_fn(batch):

    # -------------------------------------------------------------
    # 1. Receive the data
    xs = []
    ys = []
    for sample in batch:
        xs.append(sample[0])
        ys.append(sample[1])


    # or

    #xs = [item[0] for item in batch]
    #ys = [item[1] for item in batch]

    # or

    #xs, ys = zip(*batch)
    # -------------------------------------------------------------
    # -------------------------------------------------------------
    # 2. PreProcessing GET THE MAX SIZE
    l = []
    for x in xs:
        l.append(len(x))

    print(l)

    l = max(l)

    print(l)
    print()

    # -------------------------------------------------------------
    # 3. PreProcessing CREATE THE MASK and PAD
    x_pad = np.full((len(xs), l), 0, dtype=np.long) # Here its to create a matrix of the token of padding, in this case i use 0
    mask = np.zeros((len(xs), l), dtype=np.long)

    print(f'Matrix of padding before:\n{x_pad}')
    print(f'Matrix of mask before:\n{mask}')
    print()

    # -------------------------------------------------------------
    # 4. PreProcessing APPLY the PAD and MASK in the data
    for i, x in enumerate(xs):
        x_pad[i, :len(x)] = x
        mask[i, :len(x)] = 1

    print(f'Matrix of padding after:\n{x_pad}')
    print(f'Matrix of mask after:\n{mask}')
    print()
    
    # DONT FORGET TO TRANSFORM INTO TENSOR 
    # x_pad = torch.tensor(x_pax)
    # y = torch.tensor(ys)
    # mask = torch.tensor(mask)

    return {
        'x': x_pad,
        'y': ys,
        'mask': mask
    }






batch = [
    ([5, 8, 2],       1),     # sample 1: (input_ids, label)
    ([7, 1, 9, 4, 6], 0),     # sample 2: (input_ids, label)
    ([3, 3],          1),     # sample 3: (input_ids, label)
]


loader = collate_fn(batch)


print(f'Data before: {batch}')
print(f'Data after: {loader}')


[3, 5, 2]
5

Matrix of padding before:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Matrix of mask before:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Matrix of padding after:
[[5 8 2 0 0]
 [7 1 9 4 6]
 [3 3 0 0 0]]
Matrix of mask after:
[[1 1 1 0 0]
 [1 1 1 1 1]
 [1 1 0 0 0]]

Data before: [([5, 8, 2], 1), ([7, 1, 9, 4, 6], 0), ([3, 3], 1)]
Data after: {'x': array([[5, 8, 2, 0, 0],
       [7, 1, 9, 4, 6],
       [3, 3, 0, 0, 0]], dtype=int32), 'y': [1, 0, 1], 'mask': array([[1, 1, 1, 0, 0],
       [1, 1, 1, 1, 1],
       [1, 1, 0, 0, 0]], dtype=int32)}


# 2. Truncation

In [45]:
import numpy as np

def collate_fn(batch):

    # -------------------------------------------------------------
    # 1. Receive the data
    xs = []
    ys = []
    for sample in batch:
        xs.append(sample[0])
        ys.append(sample[1])


    # -------------------------------------------------------------
    # 2. PreProcessing Truncation
    seq_len = 4
    xs_trunc = []
    for x in xs:
  
        xs_trunc.append(x[:seq_len])
    xs = xs_trunc

    # -------------------------------------------------------------
    # 2. PreProcessing Padding
    L = []
    for x in xs:
        L.append(len(x))
    L = max(L)

    x_pad = np.full((len(xs), L), 0, dtype=np.long) # Here its to create a matrix of the token of padding, in this case i use 0
    mask = np.zeros((len(xs), L), dtype=np.long)

    for i, x in enumerate(xs):
            x_pad[i, :len(x)] = x
            mask[i, :len(x)] = 1

    
    # DONT FORGET TO TRANSFORM INTO TENSOR 
    # x_pad = torch.tensor(x_pax)
    # y = torch.tensor(ys)
    # mask = torch.tensor(mask)

    return {
        'x': x_pad,
        'y': ys,
        'mask': mask
    }



batch = [
    ([5, 8, 2, 4, 3, 5], 1),     # sample 1: (input_ids, label)
    ([7, 1, 9, 4, 6], 0),     # sample 2: (input_ids, label)
    ([3, 3],          1),     # sample 3: (input_ids, label)
]


loader = collate_fn(batch)


print(f'Data before: {batch}')
print(f'Data after: {loader}')


Data before: [([5, 8, 2, 4, 3, 5], 1), ([7, 1, 9, 4, 6], 0), ([3, 3], 1)]
Data after: {'x': array([[5, 8, 2, 4],
       [7, 1, 9, 4],
       [3, 3, 0, 0]], dtype=int32), 'y': [1, 0, 1], 'mask': array([[1, 1, 1, 1],
       [1, 1, 1, 1],
       [1, 1, 0, 0]], dtype=int32)}
